In [38]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from typing import TypedDict
from dotenv import load_dotenv
import os

In [53]:
load_dotenv()
api_key = os.getenv('HUGGINGFACE_API_KEY')
if not api_key:
    raise ValueError("HUGGINGFACE_API_KEY not found in .env file. Please add your API key.")
print(f"API key loaded: {api_key[:10]}...")


API key loaded: hf_OnJvVQX...


In [58]:
print(model.invoke("Say hello").content)

Hello! How can I assist you today?


In [57]:
llm = HuggingFaceEndpoint(
repo_id='Qwen/Qwen2.5-7B-Instruct',
    huggingfacehub_api_token=api_key
) 

model = ChatHuggingFace(llm=llm)

In [41]:
# define state 
class BlogApp(TypedDict):
     title : str
     brief_summary:str
     blog:str



In [42]:
# define graph
graph = StateGraph(BlogApp)

In [47]:
# create nodes functions 

def Summary(state : BlogApp)->BlogApp:
# fetch title from state 
 title = state['title']
 prompt = f'provide detailed summary of topic {title}'
 summary = model.invoke(prompt).content
 state['brief_summary'] = summary
 return state

# pass to llm and get brief summary
def createBlog(state : BlogApp)->BlogApp:
 title = state['title']
 brief_summary = state['brief_summary']
 prompt = f'create  of blog on topic {title} with brief summary on {brief_summary}'
 blog = model.invoke(prompt).content
 state['blog'] = blog
 return state


In [44]:
# define nodes
graph.add_node('Summary',Summary)
graph.add_node('createBlog', createBlog)

# define edges
graph.add_edge(START,'Summary')
graph.add_edge('Summary','createBlog')
graph.add_edge('createBlog',END)

In [46]:
workflow = graph.compile()

In [ ]:
question = ({'title':'create blog on first principle thinking'})

result = workflow.invoke(question)
print(result['blog'])
print(result)



KeyError: 'createBlog'